# BIST-Analyst: fine-tuning on Google Colab (free T4)

**Before you start:** `Runtime → Change runtime type → T4 GPU`. Run the cells top to bottom.

Everything important (checkpoints, results, the trained adapter) is written to **Google Drive**, so a disconnect does not lose work:
just reconnect, run cells 1–3 again and continue with `--resume`.

Default model: `Qwen/Qwen2.5-3B-Instruct` (fits comfortably in a free session). For the 7B model change `MODEL` in cell 3.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN = '/content/drive/MyDrive/bist_llm_run'      # all outputs go here
import os; os.makedirs(RUN, exist_ok=True); print(RUN)

## 2. Get the code and install dependencies

In [ ]:
%cd /content
!rm -rf repo && git clone -q https://github.com/yarenerisen/bist100-agentic-financial-harness.git repo
%cd /content/repo
# use the feature branch until the pull request is merged
!git checkout -q feature/tools-dataset-harness 2>/dev/null && echo "on feature branch" || echo "on main"
%cd /content/repo/bist_llm
!pip install -q -U "transformers>=4.45" "peft>=0.13" "bitsandbytes>=0.44" accelerate pyarrow liger-kernel
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Settings

In [ ]:
MODEL = 'Qwen/Qwen2.5-3B-Instruct'   # or 'Qwen/Qwen2.5-7B-Instruct' (slower; ~1-1.5 h per epoch on T4)
TAG = MODEL.split('/')[-1].lower()
ADAPTER = f'{RUN}/{TAG}-lora'          # LoRA adapter + checkpoints (on Drive)
MERGED  = f'/content/{TAG}-merged'      # merged model (local disk, re-create after a disconnect)
RESULTS = f'{RUN}/results'
FOURBIT = '--load-4bit' if '7B' in MODEL else ''
print(MODEL, ADAPTER, RESULTS)

## 4. Baseline: evaluate the BASE model (needed for the report: base vs fine-tuned)

In [ ]:
!python -m harness.evaluate --backend hf --model {MODEL} {FOURBIT} --limit 30 --tag base_E --mode E --out-dir {RESULTS}

## 5. Smoke test (30 examples, a few minutes) — checks that training runs before the long job

In [ ]:
!python training/finetune_qlora.py --model {MODEL} --out /content/smoke --limit 30 --epochs 1 --save-steps 1000 --liger

## 6. Full training (saves a checkpoint to Drive every 20 steps)
If the session disconnects: run cells 1–3 again, then run this cell again — `--resume` continues from the last checkpoint.

In [ ]:
!python training/finetune_qlora.py --model {MODEL} --out {ADAPTER} --epochs 2 --resume --liger

## 7. Evaluate the fine-tuned model

In [ ]:
# Evaluate straight from the LoRA adapter (no merge needed; works for 7B too)
!python -m harness.evaluate --backend hf --model {ADAPTER} {FOURBIT} --tag finetuned_E --mode E --out-dir {RESULTS}

## 8. Reproducibility (same question 3 times) and harness ablation A–E

In [ ]:
!python -m harness.evaluate --backend hf --model {ADAPTER} {FOURBIT} --limit 15 --repeats 3 --tag finetuned_repro --out-dir {RESULTS}
for m in 'ABCD':
    !python -m harness.evaluate --backend hf --model {ADAPTER} {FOURBIT} --limit 20 --mode {m} --tag finetuned_{m} --out-dir {RESULTS}
for m in 'AB':
    !python -m harness.evaluate --backend hf --model {MODEL} {FOURBIT} --limit 20 --mode {m} --tag base_{m} --out-dir {RESULTS}

## 9. Results table for the report

In [ ]:
import json, glob, pandas as pd
rows = []
for f in sorted(glob.glob(f'{RESULTS}/eval_*.json')):
    d = json.load(open(f)); rows.append({'run': d['tag'], **d['overall']})
pd.DataFrame(rows).set_index('run')

## 10. Ask a question (live demo)

In [ ]:
!python -m harness.run_agent --backend hf --model {ADAPTER} {FOURBIT} "In the retail sector (BIMAS, MGROS, MAVI), which stock lagged the sector median the most over the last 20 trading days up to 10.09.2026? Can it be labeled POTENTIAL_CATCH_UP_CANDIDATE?"

## 11. (Optional) Export to GGUF for running on a laptop with llama.cpp / Ollama
The file is saved to Drive; download it and follow `bist_llm/training/RUN_ON_KAGGLE.md` step 6.

In [ ]:
# merging needs ~2x model size in RAM: fine for 3B on free Colab, too large for 7B
!python training/finetune_qlora.py --merge --model {MODEL} --out {ADAPTER} --merged-out {MERGED}
!git clone -q https://github.com/ggerganov/llama.cpp /content/llama.cpp && pip install -q -r /content/llama.cpp/requirements.txt
!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED} --outfile {RUN}/bist-analyst-{TAG}-q8_0.gguf --outtype q8_0